# 06 - Document AI

This notebook demonstrates OCI Vision's document processing capabilities.
Document AI extracts structured data from documents: fields, tables, and
key-value pairs. Since demo data is not cached for this feature, we show
the API pattern and how results map to a pandas DataFrame.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient

client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

The `analyze_document()` method calls OCI's Document AI service.
In demo mode this returns `None` because no document demo data is
cached yet. We show the call and then build synthetic examples.

In [ ]:
result = client.analyze_document("dog_closeup.jpg")
print(f"analyze_document() returned: {result}")
print()
print("In live mode, this returns a DocumentResult containing")
print("extracted fields, tables, and key-value pairs.")

## Explore the results

We construct a synthetic `DocumentResult` to demonstrate the data
model -- fields (key-value pairs) and tables.

In [ ]:
from oci_vision.core.models import DocumentResult, DocumentField, DocumentTable

# Synthetic example -- an invoice document
example_result = DocumentResult(
    model_version="1.5.97",
    fields=[
        DocumentField(field_type="KEY_VALUE", label="Invoice Number",
                      value="INV-2024-0042", confidence=0.9845),
        DocumentField(field_type="KEY_VALUE", label="Date",
                      value="2024-03-15", confidence=0.9723),
        DocumentField(field_type="KEY_VALUE", label="Vendor",
                      value="Acme Corp", confidence=0.9912),
        DocumentField(field_type="KEY_VALUE", label="Total Amount",
                      value="$1,234.56", confidence=0.9567),
        DocumentField(field_type="KEY_VALUE", label="Tax",
                      value="$123.46", confidence=0.9234),
        DocumentField(field_type="KEY_VALUE", label="Due Date",
                      value="2024-04-15", confidence=0.9678),
    ],
    tables=[
        DocumentTable(
            row_count=4,
            column_count=4,
            header_rows=["Item", "Qty", "Unit Price", "Total"],
            body_rows=[
                ["Widget A", "10", "$50.00", "$500.00"],
                ["Widget B", "5", "$100.00", "$500.00"],
                ["Service Fee", "1", "$234.56", "$234.56"],
            ],
            confidence=0.9456,
        )
    ]
)

print(f"Fields extracted: {len(example_result.fields)}")
print(f"Tables found:     {len(example_result.tables)}")

### Extracted fields (key-value pairs)

In [ ]:
# Display extracted fields
print(f"{'Field':20s}  {'Value':20s}  {'Confidence':>10s}")
print('-' * 55)
for field in example_result.fields:
    print(f"{field.label:20s}  {field.value:20s}  {field.confidence:9.2%}")

### Extracted tables

In [ ]:
# Display tables
for t_idx, table in enumerate(example_result.tables, 1):
    print(f"Table {t_idx}: {table.row_count} rows x {table.column_count} cols "
          f"(confidence: {table.confidence:.1%})")
    print()
    # Header
    header = table.header_rows
    print(f"  {'  '.join(f'{h:>12s}' for h in header)}")
    print(f"  {'  '.join('-' * 12 for _ in header)}")
    # Body
    for row in table.body_rows:
        print(f"  {'  '.join(f'{c:>12s}' for c in row)}")

### Convert to pandas DataFrame

In [ ]:
import pandas as pd

# Fields -> DataFrame
fields_df = pd.DataFrame([
    {"Field": f.label, "Value": f.value, "Confidence": f"{f.confidence:.1%}"}
    for f in example_result.fields
])
print("Key-Value Pairs:")
print(fields_df.to_string(index=False))
print()

# Table -> DataFrame
for t_idx, table in enumerate(example_result.tables, 1):
    table_df = pd.DataFrame(table.body_rows, columns=table.header_rows)
    print(f"Table {t_idx}:")
    print(table_df.to_string(index=False))

## Visualize

For document AI, visualization typically means highlighting detected
fields and table cells on the original document image. The renderer
handles this when document data is available in an `AnalysisReport`.
Here we show the fields as a simple chart.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Confidence by field
labels = [f.label for f in example_result.fields]
confs = [f.confidence * 100 for f in example_result.fields]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, confs, color='#2A9D8F')
ax.set_xlabel('Confidence (%)')
ax.set_title('Document Field Extraction Confidence')
ax.set_xlim(0, 105)
for i, (label, conf) in enumerate(zip(labels, confs)):
    ax.text(conf + 0.5, i, f'{conf:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('document_field_confidence.png', dpi=100, bbox_inches='tight')
plt.show()

## Under the hood

In [ ]:
import json

raw = example_result.model_dump()
print(json.dumps(raw, indent=2, default=str))

### Live API usage

```python
client = VisionClient()  # uses ~/.oci/config
result = client.analyze_document("invoice.pdf")

# Extract key-value pairs
for field in result.fields:
    print(f"{field.label}: {field.value} ({field.confidence:.1%})")

# Extract tables
for table in result.tables:
    df = pd.DataFrame(table.body_rows, columns=table.header_rows)
    print(df)
```

## Try it yourself

1. **Invoice processing**: Use `VisionClient()` with real credentials
   to extract fields from scanned invoices or receipts.
2. **Table extraction**: Process documents with tabular data and convert
   the extracted tables directly into pandas DataFrames.
3. **Batch processing**: Loop over multiple document images, extract
   fields, and aggregate results into a single DataFrame.
4. **Confidence filtering**: Skip low-confidence fields to reduce noise
   in automated pipelines.